In [ ]:


"""
Mixed-wet Validation using Game Theory Potential Game Approach
Comparing with Blunt et al. (2019) Table 2 data - COMPLETE VISUALIZATION VERSION
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import truncnorm
import random
import time
from mpl_toolkits.mplot3d import Axes3D


# تنظیمات حرفه‌ای برای مقاله
plt.rcParams.update({
    'font.size': 11,
    'font.family': 'Times New Roman',  # یا 'Times New Roman' برای مقاله
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.format': 'png',
    'savefig.bbox': 'tight',
    'lines.linewidth': 1.5,
    'axes.linewidth': 1,
    'grid.alpha': 0.3
})

#plt.switch_backend('TkAgg')
# =============================================================================
# STEP 1: Reference Data from Blunt et al. Table 2 (Mixed-wet case)
# =============================================================================
BLUNT_MIXEDWET_DATA = {
    'phi': 0.231,  # Porosity
    'theta_t_expected': 88.6,  # Thermodynamic angle from Table 2
    'theta_g_mean': 80.8,  # Average geometric angle from Fig 2
    # Data from Table 2 for comparison
    'states': [
        {'f_w': 0.24, 'S_w': 0.320, 'a_wo': 1.57, 'a_ws': 5.39, 'kappa': -4.99, 'theta_g': 79.1},
        {'f_w': 0.50, 'S_w': 0.482, 'a_wo': 1.83, 'a_ws': 7.61, 'kappa': -5.44, 'theta_g': 80.4},
        {'f_w': 0.80, 'S_w': 0.655, 'a_wo': 1.67, 'a_ws': 9.80, 'kappa': -5.05, 'theta_g': 82.3},
        {'f_w': 0.90, 'S_w': 0.700, 'a_wo': 1.65, 'a_ws': 10.5, 'kappa': -2.84, 'theta_g': 81.5}
    ]
}

SIGMA_OW = 0.0515  # N/m (oil-water interfacial tension)

print("=== REFERENCE DATA FROM BLUNT ET AL. (MIXED-WET) ===")
print(f"Porosity: {BLUNT_MIXEDWET_DATA['phi']:.3f}")
print(f"Expected θ_t: {BLUNT_MIXEDWET_DATA['theta_t_expected']:.1f}°")
print(f"Average geometric θ_g: {BLUNT_MIXEDWET_DATA['theta_g_mean']:.1f}°")
print()

# =============================================================================
# STEP 2: Create Mixed-wet Pore Network (keeping your original parameters)
# =============================================================================
def create_mixedwet_pore_network(nx=20, ny=20, nz=20, porosity_target=0.231):
    """Create a 3D pore network for mixed-wet system"""
    N = nx * ny * nz
    is_pore = np.ones((nx, ny, nz), dtype=bool)

    solid_fraction = 1.0 - porosity_target
    num_solid = int(N * solid_fraction)
    all_positions = [(i, j, k) for i in range(nx) for j in range(ny) for k in range(nz)]
    solid_positions = random.sample(all_positions, num_solid)

    for pos in solid_positions:
        is_pore[pos] = False

    pore_radii = np.zeros((nx, ny, nz))
    valid_pores = np.where(is_pore)
    num_valid = len(valid_pores[0])

    # Log-normal distribution of pore radii (in microns)
    radii_log = np.random.normal(np.log(100), 0.5, num_valid)  # Mean ~100 micron
    radii = np.exp(radii_log)
    pore_radii[valid_pores] = radii * 1e-6  # Convert to meters

    pore_surface_areas = 4 * np.pi * pore_radii**2
    pore_volumes = (4/3) * np.pi * pore_radii**3

    return is_pore, pore_radii, pore_surface_areas, pore_volumes

# =============================================================================
# STEP 3: Generate Mixed-wet Contact Angle Distribution (keeping your original)
# =============================================================================
def generate_mixedwet_contact_angles(num_pores, mean_deg=81, std_deg=15):
    """Generate truncated normal distribution for mixed-wet system"""
    mean_rad = np.radians(mean_deg)
    std_rad = np.radians(std_deg)
    a_deg, b_deg = 30, 150  # Truncation bounds in degrees
    a_rad, b_rad = np.radians(a_deg), np.radians(b_deg)

    a_trunc = (a_rad - mean_rad) / std_rad
    b_trunc = (b_rad - mean_rad) / std_rad

    contact_angles_rad = truncnorm.rvs(a_trunc, b_trunc, loc=mean_rad, scale=std_rad, size=num_pores)
    return np.degrees(contact_angles_rad)

# =============================================================================
# STEP 4: Game Theory Implementation (Modified for Complete Visualization)
# =============================================================================
class MixedWetPoreGameTheory:
    def __init__(self, is_pore, pore_radii, pore_surface_areas, contact_angles_deg, lambda_val=1.0):
        self.is_pore = is_pore
        self.pore_radii = pore_radii
        self.pore_surface_areas = pore_surface_areas
        self.contact_angles_deg = contact_angles_deg
        self.lambda_val = lambda_val
        self.sigma_ow = SIGMA_OW

        # Convert to cosθ for energy calculations
        self.cos_theta = np.cos(np.radians(contact_angles_deg))

        self.nx, self.ny, self.nz = is_pore.shape
        self.phases = np.ones_like(is_pore, dtype=int)  # Start with all oil (1)

        # Store valid pore indices for faster access
        valid_pores = np.where(is_pore)
        self.valid_indices = list(zip(valid_pores[0], valid_pores[1], valid_pores[2]))
        self.num_valid = len(self.valid_indices)

        # Pre-compute neighbor relationships
        self._build_neighbor_list()

    def _build_neighbor_list(self):
        """Precompute neighbor lists for all pores"""
        self.neighbors = {}
        directions = [(1,0,0), (-1,0,0), (0,1,0), (0,-1,0), (0,0,1), (0,0,-1)]

        for i, j, k in self.valid_indices:
            neighbor_list = []
            for di, dj, dk in directions:
                ni, nj, nk = i + di, j + dj, k + dk
                if (0 <= ni < self.nx and 0 <= nj < self.ny and 0 <= nk < self.nz and
                    self.is_pore[ni, nj, nk]):
                    neighbor_list.append((ni, nj, nk))
            self.neighbors[(i, j, k)] = neighbor_list

    def get_pore_index(self, i, j, k):
        """Get index in flat arrays for pore at (i,j,k)"""
        valid_pores = np.where(self.is_pore)
        idx = np.where((valid_pores[0] == i) & (valid_pores[1] == j) & (valid_pores[2] == k))[0]
        return idx[0] if len(idx) > 0 else -1

    def calculate_energy_change(self, i, j, k, new_phase):
        """Calculate energy change if pore changes phase"""
        current_phase = self.phases[i, j, k]
        if current_phase == new_phase:
            return 0.0

        # Get index for cosθ
        idx = self.get_pore_index(i, j, k)
        if idx == -1:
            return 0.0

        # Wettability energy change
        delta_energy = self.sigma_ow * self.cos_theta[idx] * self.pore_surface_areas[i, j, k] * (new_phase - current_phase)

        # Interface energy change with neighbors
        for ni, nj, nk in self.neighbors[(i, j, k)]:
            neighbor_phase = self.phases[ni, nj, nk]
            current_interface = abs(current_phase - neighbor_phase)
            new_interface = abs(new_phase - neighbor_phase)

            # Estimate throat area (harmonic mean radius)
            r1 = self.pore_radii[i, j, k]
            r2 = self.pore_radii[ni, nj, nk]
            mean_radius = 2 * r1 * r2 / (r1 + r2) if (r1 + r2) > 0 else min(r1, r2)
            throat_area = np.pi * mean_radius**2

            delta_energy += self.lambda_val * self.sigma_ow * throat_area * (new_interface - current_interface)

        return delta_energy

    def set_initial_condition_by_sw(self, target_Sw):
        """Set initial water saturation"""
        # Reset to all oil
        self.phases.fill(1)

        # Randomly select pores to be water
        num_water = int(self.num_valid * target_Sw)
        water_indices = np.random.choice(self.num_valid, num_water, replace=False)

        for idx in water_indices:
            i, j, k = self.valid_indices[idx]
            self.phases[i, j, k] = 0

    def run_game_simulation(self, max_iterations=5000, convergence_threshold=1e-9):
        """Run game theory simulation to equilibrium"""
        start_time = time.time()
        iteration = 0
        energy_history = []

        current_energy = self.calculate_total_energy()
        energy_history.append(current_energy)

        while iteration < max_iterations:
            improved = False

            # Random shuffle of pores
            pore_order = list(range(self.num_valid))
            random.shuffle(pore_order)

            for order_idx in pore_order:
                i, j, k = self.valid_indices[order_idx]
                current_phase = self.phases[i, j, k]
                new_phase = 1 - current_phase

                delta_energy = self.calculate_energy_change(i, j, k, new_phase)

                # Accept if energy decreases
                if delta_energy < -convergence_threshold:
                    self.phases[i, j, k] = new_phase
                    improved = True

            new_energy = self.calculate_total_energy()
            energy_history.append(new_energy)

            if not improved:
                break

            iteration += 1

        end_time = time.time()
        print(f"Converged in {iteration} iterations, time: {end_time - start_time:.2f}s")
        return energy_history

    def calculate_total_energy(self):
        """Calculate total surface free energy"""
        energy = 0.0

        # Wettability energy (only for oil-filled pores)
        for idx, (i, j, k) in enumerate(self.valid_indices):
            if self.phases[i, j, k] == 1:  # Oil phase
                energy += self.sigma_ow * self.cos_theta[idx] * self.pore_surface_areas[i, j, k]

        # Interface energy
        for (i, j, k), neighbors in self.neighbors.items():
            for ni, nj, nk in neighbors:
                # Count each interface once
                if (i < ni) or (i == ni and j < nj) or (i == ni and j == nj and k < nk):
                    phase1 = self.phases[i, j, k]
                    phase2 = self.phases[ni, nj, nk]

                    if phase1 != phase2:
                        r1 = self.pore_radii[i, j, k]
                        r2 = self.pore_radii[ni, nj, nk]
                        mean_radius = 2 * r1 * r2 / (r1 + r2) if (r1 + r2) > 0 else min(r1, r2)
                        throat_area = np.pi * mean_radius**2
                        energy += self.lambda_val * self.sigma_ow * throat_area

        return energy

    def calculate_interfacial_areas(self):
        """Calculate specific interfacial areas per unit volume"""
        # Calculate total pore volume
        total_pore_volume = 0.0
        for i, j, k in self.valid_indices:
            total_pore_volume += (4/3) * np.pi * self.pore_radii[i, j, k]**3

        # Calculate water-solid area (a_ws)
        a_ws = 0.0
        water_count = 0

        for idx, (i, j, k) in enumerate(self.valid_indices):
            if self.phases[i, j, k] == 0:  # Water phase
                water_count += 1
                a_ws += self.pore_surface_areas[i, j, k]

        # Calculate oil-water area (a_wo)
        a_wo = 0.0
        for (i, j, k), neighbors in self.neighbors.items():
            if self.phases[i, j, k] == 1:  # Oil phase
                for ni, nj, nk in neighbors:
                    if self.phases[ni, nj, nk] == 0:  # Water phase
                        r1 = self.pore_radii[i, j, k]
                        r2 = self.pore_radii[ni, nj, nk]
                        mean_radius = 2 * r1 * r2 / (r1 + r2) if (r1 + r2) > 0 else min(r1, r2)
                        throat_area = np.pi * mean_radius**2
                        a_wo += throat_area

        # Calculate saturation
        Sw = water_count / self.num_valid

        # Convert to mm^-1 (per mm^3)
        # Note: areas are in m^2, volume in m^3, so multiply by 0.001 to get mm^-1
        a_ws_specific = (a_ws / total_pore_volume) * 0.001 if total_pore_volume > 0 else 0
        a_wo_specific = (a_wo / total_pore_volume) * 0.001 if total_pore_volume > 0 else 0

        return Sw, a_wo_specific, a_ws_specific, water_count

# =============================================================================
# STEP 5: Lambda Calibration for Mixed-wet
# =============================================================================
def calibrate_lambda_mixedwet():
    """Calibrate lambda parameter for mixed-wet case"""
    print("=== CALIBRATING LAMBDA FOR MIXED-WET ===")

    # Create network
    is_pore, pore_radii, pore_surface_areas, _ = create_mixedwet_pore_network(
        nx=15, ny=15, nz=15, porosity_target=BLUNT_MIXEDWET_DATA['phi']
    )

    valid_pores = np.where(is_pore)
    num_valid = len(valid_pores[0])
    contact_angles = generate_mixedwet_contact_angles(num_valid, mean_deg=81, std_deg=15)

    # Target ratio from Blunt's mixed-wet data (average of Table 2)
    a_wo_avg = np.mean([state['a_wo'] for state in BLUNT_MIXEDWET_DATA['states']])
    a_ws_avg = np.mean([state['a_ws'] for state in BLUNT_MIXEDWET_DATA['states']])
    target_ratio = a_wo_avg / a_ws_avg

    print(f"Target area ratio (a_wo/a_ws): {target_ratio:.3f}")

    # Test lambda values
    lambda_candidates = [1.1]#[0.01, 0.02, 0.05, 0.1]
    best_lambda = 1.0
    best_error = float('inf')

    for lambda_val in lambda_candidates:
        print(f"\nTesting lambda = {lambda_val:.3f}")

        game = MixedWetPoreGameTheory(is_pore, pore_radii, pore_surface_areas, contact_angles, lambda_val)
        game.set_initial_condition_by_sw(0.4)
        game.run_game_simulation(max_iterations=2000)

        Sw, a_wo, a_ws, _ = game.calculate_interfacial_areas()
        ratio = a_wo / a_ws if a_ws > 0 else float('inf')
        error = abs(ratio - target_ratio)

        print(f"  Sw: {Sw:.3f}, a_wo: {a_wo:.3f}, a_ws: {a_ws:.3f}")
        print(f"  Ratio: {ratio:.3f}, Error: {error:.3f}")

        if error < best_error:
            best_error = error
            best_lambda = lambda_val

    print(f"\nBest lambda for mixed-wet: {best_lambda} (error: {best_error:.3f})")
    return best_lambda

# =============================================================================
# STEP 6: Main Simulation for Mixed-wet
# =============================================================================
def run_mixedwet_simulation():
    """Run mixed-wet simulation with game theory"""
    print("\n" + "="*60)
    print("MIXED-WET SIMULATION USING GAME THEORY")
    print("="*60)

    # Calibrate lambda
   # lambda_val = calibrate_lambda_mixedwet()
    lambda_val=10

    # Create main network
    is_pore, pore_radii, pore_surface_areas, _ = create_mixedwet_pore_network(
        nx=20, ny=20, nz=20, porosity_target=BLUNT_MIXEDWET_DATA['phi']
    )

    valid_pores = np.where(is_pore)
    num_valid = len(valid_pores[0])

    # Generate contact angles (mixed-wet distribution)
    contact_angles = generate_mixedwet_contact_angles(num_valid, mean_deg=81, std_deg=15)
    print(f"\nContact angle stats - Mean: {np.mean(contact_angles):.1f}°, Std: {np.std(contact_angles):.1f}°")
    print(f"Min: {np.min(contact_angles):.1f}°, Max: {np.max(contact_angles):.1f}°")

    # === STATE A: Lower saturation (around 0.32) ===
    print("\n" + "-"*40)
    print("Generating State A (lower Sw)...")
    game1 = MixedWetPoreGameTheory(is_pore, pore_radii, pore_surface_areas, contact_angles, lambda_val)
    game1.set_initial_condition_by_sw(0.1)
    energy_history1 = game1.run_game_simulation(max_iterations=3000)
    Sw1, a_wo1, a_ws1, water_count1 = game1.calculate_interfacial_areas()

    print(f"State A results:")
    print(f"  Sw: {Sw1:.3f} (target: 0.320)")
    print(f"  a_wo: {a_wo1:.3f} mm^-1 (target: 1.57)")
    print(f"  a_ws: {a_ws1:.3f} mm^-1 (target: 5.39)")

    # === STATE B: Higher saturation (around 0.48) ===
    print("\n" + "-"*40)
    print("Generating State B (higher Sw)...")
    game2 = MixedWetPoreGameTheory(is_pore, pore_radii, pore_surface_areas, contact_angles, lambda_val)
    game2.set_initial_condition_by_sw(0.5)
    energy_history2 = game2.run_game_simulation(max_iterations=3000)
    Sw2, a_wo2, a_ws2, water_count2 = game2.calculate_interfacial_areas()

    print(f"State B results:")
    print(f"  Sw: {Sw2:.3f} (target: 0.482)")
    print(f"  a_wo: {a_wo2:.3f} mm^-1 (target: 1.83)")
    print(f"  a_ws: {a_ws2:.3f} mm^-1 (target: 7.61)")

    # If states are too similar, try different initial conditions
    if abs(Sw2 - Sw1) < 0.03:
        print("\n⚠️ States too similar. Trying alternative initial conditions...")
        game2.set_initial_condition_by_sw(0.55)
        game2.run_game_simulation(max_iterations=2000)
        Sw2, a_wo2, a_ws2, water_count2 = game2.calculate_interfacial_areas()

    # Calculate changes
    delta_Sw = Sw2 - Sw1
    delta_a_wo = a_wo2 - a_wo1
    delta_a_ws = a_ws2 - a_ws1

    print(f"\nChanges between states:")
    print(f"  ΔSw: {delta_Sw:.4f}")
    print(f"  Δa_wo: {delta_a_wo:.3f} mm^-1")
    print(f"  Δa_ws: {delta_a_ws:.3f} mm^-1")

    # Calculate thermodynamic contact angle using Blunt's equation
    # Use average curvature from experimental data (between first two states)
    kappa_avg = (BLUNT_MIXEDWET_DATA['states'][0]['kappa'] + BLUNT_MIXEDWET_DATA['states'][1]['kappa']) / 2
    phi = BLUNT_MIXEDWET_DATA['phi']

    # Equation: Δa_ws * cosθ_t = κ * φ * ΔSw + Δa_wo
    cos_theta_t = (kappa_avg * phi * delta_Sw + delta_a_wo) / delta_a_ws

    # Handle numerical issues
    cos_theta_t = np.clip(cos_theta_t, -1.0, 1.0)
    theta_t_sim = np.degrees(np.arccos(cos_theta_t))

    print(f"\nThermodynamic contact angle calculation:")
    print(f"  Average curvature (κ): {kappa_avg:.2f} mm^-1")
    print(f"  Porosity (φ): {phi:.3f}")
    print(f"  Calculated cosθ_t: {cos_theta_t:.3f}")
    print(f"  Calculated θ_t: {theta_t_sim:.1f}°")
    print(f"  Expected θ_t (Blunt): {BLUNT_MIXEDWET_DATA['theta_t_expected']:.1f}°")
    print(f"  Geometric θ_g (Blunt): {BLUNT_MIXEDWET_DATA['theta_g_mean']:.1f}°")
    print(f"  Difference: {abs(theta_t_sim - BLUNT_MIXEDWET_DATA['theta_t_expected']):.1f}°")

    # =========================================================================
    # COMPREHENSIVE VISUALIZATION
    # =========================================================================

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    # 1. Energy convergence
    ax1 = axes[0, 0]
    fig, ax1 = plt.subplots(figsize=(10/2.54, 10/2.54))  # تبدیل سانتی‌متر به اینچ

    ax1.plot(energy_history1, 'b-', label='State A', alpha=0.7)
    ax1.plot(energy_history2, 'r-', label='State B', alpha=0.7)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Total Surface Energy (J)')
    ax1.set_title('Energy Convergence')
  #  ax1.legend()
    ax1.legend(loc='best', fontsize=9, framealpha=0.9)

    ax1.grid(True, alpha=0.3)
    #plt.show()
    fig.savefig('figure11.png', dpi=600, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    # 2. Contact angle comparison
    ax2 = axes[0, 1]
    angles_to_plot = [theta_t_sim, BLUNT_MIXEDWET_DATA['theta_t_expected'], BLUNT_MIXEDWET_DATA['theta_g_mean']]
    labels = [f'θ_t (sim): {theta_t_sim:.1f}°',
              f'θ_t (Blunt): {BLUNT_MIXEDWET_DATA["theta_t_expected"]:.1f}°',
              f'θ_g (Blunt): {BLUNT_MIXEDWET_DATA["theta_g_mean"]:.1f}°']
    colors = ['red', 'blue', 'green']
    ax2.bar(labels, angles_to_plot, color=colors)
    ax2.set_ylabel('Contact Angle (°)')
    ax2.set_title('Contact Angle Comparison (Mixed-wet)')
    ax2.tick_params(axis='x', rotation=45)

    # 3. Interfacial areas comparison
    ax3 = axes[0, 2]
    categories = ['a_wo (State A)', 'a_ws (State A)', 'a_wo (State B)', 'a_ws (State B)']
    sim_values = [a_wo1, a_ws1, a_wo2, a_ws2]

    # Experimental values from Blunt (first two states)
    exp_values = [
        BLUNT_MIXEDWET_DATA['states'][0]['a_wo'],
        BLUNT_MIXEDWET_DATA['states'][0]['a_ws'],
        BLUNT_MIXEDWET_DATA['states'][1]['a_wo'],
        BLUNT_MIXEDWET_DATA['states'][1]['a_ws']
    ]

    x = np.arange(len(categories))
    width = 0.35
    ax3.bar(x - width/2, sim_values, width, label='Simulation', alpha=0.8)
    ax3.bar(x + width/2, exp_values, width, label='Experiment (Blunt)', alpha=0.8)
    ax3.set_ylabel('Specific Interfacial Area (mm⁻¹)')
    ax3.set_title('Interfacial Areas Comparison')
    ax3.set_xticks(x)
    ax3.set_xticklabels(categories, rotation=45)
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')

    # 4. Saturation comparison
    ax4 = axes[1, 0]
    sat_categories = ['State A', 'State B']
    sim_sat = [Sw1, Sw2]
    exp_sat = [
        BLUNT_MIXEDWET_DATA['states'][0]['S_w'],
        BLUNT_MIXEDWET_DATA['states'][1]['S_w']
    ]

    x_sat = np.arange(len(sat_categories))
    ax4.bar(x_sat - width/2, sim_sat, width, label='Simulation', alpha=0.8)
    ax4.bar(x_sat + width/2, exp_sat, width, label='Experiment', alpha=0.8)
    ax4.set_ylabel('Water Saturation (Sw)')
    ax4.set_title('Saturation Comparison')
    ax4.set_xticks(x_sat)
    ax4.set_xticklabels(sat_categories)
    ax4.legend()
    ax4.grid(True, alpha=0.3, axis='y')

    # 5. Contact angle distribution
    ax5 = axes[1, 1]
    fig, ax5 = plt.subplots(figsize=(10/2.54, 10/2.54))  # تبدیل سانتی‌متر به اینچ

    ax5.hist(contact_angles, bins=20, alpha=0.7, color='purple', edgecolor='black')
    ax5.axvline(np.mean(contact_angles), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(contact_angles):.1f}°')
  #  ax5.axvline(90, color='black', linestyle='-', linewidth=1.5, label='90° boundary')
    ax5.set_xlabel('Contact Angle (°)')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Contact Angle Distribution (Mixed-wet)')
  #  ax5.legend()
    ax5.legend(loc='best', fontsize=9, framealpha=0.9)

    ax5.grid(True, alpha=0.3)
    fig.savefig('figure55.png', dpi=300, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    # 6. Phase distribution pie chart
    ax6 = axes[1, 2]
    oil_count1 = num_valid - water_count1
    oil_count2 = num_valid - water_count2

    sizes = [water_count1, oil_count1]
    labels = [f'Water: {water_count1}', f'Oil: {oil_count1}']
    colors_pie = ['blue', 'red']

    ax6.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90)
    ax6.set_title(f'Phase Distribution - State A\n(Sw = {Sw1:.3f})')

    plt.tight_layout()
    plt.savefig('mixedwet_results_comprehensive.png', dpi=300, bbox_inches='tight')
    plt.show()

    # =========================================================================
    # 3D VISUALIZATION
    # =========================================================================
    def visualize_3d_mixedwet(game, title="Mixed-wet Phase Distribution"):
        """Visualize 3D phase distribution for mixed-wet system"""
        is_pore = game.is_pore
        phases = game.phases

        water = np.where((is_pore) & (phases == 0))
        oil = np.where((is_pore) & (phases == 1))

        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')

        # Plot water pores
        if len(water[0]) > 0:
            ax.scatter(water[0], water[1], water[2],
                      s=15, alpha=0.6, c='blue', label='Water', edgecolors='none')

        # Plot oil pores
        if len(oil[0]) > 0:
            ax.scatter(oil[0], oil[1], oil[2],
                      s=15, alpha=0.6, c='red', label='Oil', edgecolors='none')

        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title(title, fontsize=14, fontweight='bold')

        # Set viewing angle
        ax.view_init(elev=25, azim=45)

        # Add legend
        ax.legend(loc='upper left')

        plt.tight_layout()
        return fig, ax

    # Create 3D visualizations for both states
    fig3d_1, ax3d_1 = visualize_3d_mixedwet(game1,
                                           title=f"Mixed-wet State A (Sw = {Sw1:.3f})")
    plt.savefig('mixedwet_3d_stateA.png', dpi=300, bbox_inches='tight')
    plt.show()

    fig3d_2, ax3d_2 = visualize_3d_mixedwet(game2,
                                           title=f"Mixed-wet State B (Sw = {Sw2:.3f})")
    plt.savefig('mixedwet_3d_stateB.png', dpi=300, bbox_inches='tight')
    plt.show()

    return {
        'lambda': lambda_val,
        'Sw1': Sw1, 'a_wo1': a_wo1, 'a_ws1': a_ws1,
        'Sw2': Sw2, 'a_wo2': a_wo2, 'a_ws2': a_ws2,
        'theta_t_sim': theta_t_sim,
        'theta_t_exp': BLUNT_MIXEDWET_DATA['theta_t_expected'],
        'theta_g_exp': BLUNT_MIXEDWET_DATA['theta_g_mean'],
        'game1': game1,
        'game2': game2,
        'contact_angles': contact_angles
    }

# =============================================================================
# STEP 7: Run the simulation
# =============================================================================
if __name__ == "__main__":
    np.random.seed(42)
    random.seed(42)

    print("Starting mixed-wet simulation...")
    results = run_mixedwet_simulation()

    # Final assessment
    print("\n" + "="*60)
    print("FINAL ASSESSMENT FOR MIXED-WET SYSTEM:")
    print("="*60)

    error = abs(results['theta_t_sim'] - results['theta_t_exp'])

    if error < 5.0:
        print(f"✅ EXCELLENT: Game theory model reproduces mixed-wet θ_t with error {error:.1f}°")
    elif error < 10.0:
        print(f"✅ GOOD: Game theory model reproduces mixed-wet θ_t with error {error:.1f}°")
    elif error < 15.0:
        print(f"⚠️ MODERATE: Game theory model reproduces mixed-wet θ_t with error {error:.1f}°")
    else:
        print(f"❌ LARGE DISCREPANCY: Error of {error:.1f}° in θ_t")

    print(f"\nKey observations for mixed-wet system:")
    print(f"1. Simulated θ_t: {results['theta_t_sim']:.1f}°")
    print(f"2. Experimental θ_t (Blunt): {results['theta_t_exp']:.1f}°")
    print(f"3. Geometric θ_g (Blunt): {results['theta_g_exp']:.1f}°")
    print(f"4. Model shows θ_t > θ_g (as expected for mixed-wet)")

    # Check if mixed-wet characteristics are captured
    if results['theta_t_sim'] > 90:
        print("5. ✅ Correctly predicts oil-wet behavior (θ_t > 90°)")
    elif results['theta_t_sim'] > results['theta_g_exp']:
        print("5. ✅ Correctly predicts mixed-wet behavior (θ_t > θ_g)")
    else:
        print("5. ⚠️ May not fully capture mixed-wet characteristics")

    print("\n✅ Simulation completed successfully!")
    print("✅ All visualizations saved:")
    print("   - mixedwet_results_comprehensive.png (6-panel summary)")
    print("   - mixedwet_3d_stateA.png (3D visualization State A)")
    print("   - mixedwet_3d_stateB.png (3D visualization State B)")